In [41]:
import numpy as np
import pandas as pd

In [42]:
X_ge_train = np.load("../data/processed/GE_train_encoded.npy")
y_ge_train = np.load("../data/processed/GE_train_labels.npy")

X_ge_test = np.load("../data/processed/GE_test_encoded.npy")
y_ge_test = np.load("../data/processed/GE_test_labels.npy")

In [4]:
X_dm_train = np.load("../data/processed/DM_train_encoded.npy")
y_dm_train = np.load("../data/processed/DM_train_labels.npy")

X_dm_test = np.load("../data/processed/DM_test_encoded.npy")
y_dm_test = np.load("../data/processed/DM_test_labels.npy")

In [5]:
print("GE train:", X_ge_train.shape)
print("DM train:", X_dm_train.shape)

print("GE test:", X_ge_test.shape)
print("DM test:", X_dm_test.shape)

GE train: (487, 64)
DM train: (99, 16)
GE test: (210, 64)
DM test: (43, 16)


In [14]:
import random

X_train_combined = []
y_train_combined = []

for i in range(len(X_dm_train)):
    # find all matching GE indices
    matching_indices = [j for j in range(len(X_ge_train)) if y_ge_train[j] == y_dm_train[i]]
    
    # randomly pick one
    j = random.choice(matching_indices)
    
    combined = np.concatenate((X_ge_train[j], X_dm_train[i]))
    
    X_train_combined.append(combined)
    y_train_combined.append(y_dm_train[i])

X_train_combined = np.array(X_train_combined)
y_train_combined = np.array(y_train_combined)

In [7]:
print("X_train_combined:", X_train_combined.shape)
print("y_train_combined:", y_train_combined.shape)

X_train_combined: (99, 80)
y_train_combined: (99,)


In [8]:
print(X_train_combined.shape[1])

80


In [9]:
print(X_train_combined.shape[1])

80


In [10]:
# Check GE + DM labels for first few samples
for i in range(5):
    for j in range(len(X_ge_train)):
        if y_dm_train[i] == y_ge_train[j]:
            print("DM label:", y_dm_train[i], "GE label:", y_ge_train[j])
            break

DM label: 0 GE label: 0
DM label: 0 GE label: 0
DM label: 1 GE label: 1
DM label: 1 GE label: 1
DM label: 0 GE label: 0


In [11]:
import numpy as np

print("Class distribution:", np.unique(y_train_combined, return_counts=True))

Class distribution: (array([0, 1]), array([47, 52]))


In [15]:
print(X_train_combined[:2])

[[0.0000000e+00 7.9331508e+00 0.0000000e+00 0.0000000e+00 4.4873695e+01
  0.0000000e+00 8.1029415e-01 0.0000000e+00 1.1526417e+01 0.0000000e+00
  0.0000000e+00 0.0000000e+00 1.3967851e+01 3.1038160e+01 0.0000000e+00
  0.0000000e+00 0.0000000e+00 4.1694969e+01 0.0000000e+00 1.5863408e+01
  0.0000000e+00 3.8371408e+00 0.0000000e+00 2.8389008e+01 4.6549931e+00
  0.0000000e+00 1.0090453e+01 1.9036425e+01 0.0000000e+00 6.3689427e+00
  0.0000000e+00 7.4980788e+00 0.0000000e+00 2.4679604e+01 0.0000000e+00
  0.0000000e+00 3.0876865e+01 0.0000000e+00 0.0000000e+00 3.6123665e+01
  0.0000000e+00 5.6492434e+00 0.0000000e+00 1.7729795e+01 2.3063948e+01
  2.9406746e+01 0.0000000e+00 0.0000000e+00 2.0539871e+01 3.9604857e+00
  1.8796793e+01 2.0813213e+01 0.0000000e+00 4.7766314e+00 0.0000000e+00
  1.9433146e+01 0.0000000e+00 5.0647146e-01 0.0000000e+00 0.0000000e+00
  0.0000000e+00 4.2357445e+01 0.0000000e+00 0.0000000e+00 0.0000000e+00
  0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000000e+00 0.0000

In [13]:
print(np.isnan(X_train_combined).sum())

0


In [16]:
from sklearn.utils import resample

n_bags = 5
bags_X = []
bags_y = []

for i in range(n_bags):
    X_bag, y_bag = resample(
        X_train_combined,
        y_train_combined,
        replace=True,   # important for bagging
        n_samples=len(X_train_combined),
        random_state=i
    )
    
    bags_X.append(X_bag)
    bags_y.append(y_bag)

In [17]:
for i in range(n_bags):
    print(f"Bag {i+1} shape:", bags_X[i].shape)

Bag 1 shape: (99, 80)
Bag 2 shape: (99, 80)
Bag 3 shape: (99, 80)
Bag 4 shape: (99, 80)
Bag 5 shape: (99, 80)


In [18]:
from sklearn.svm import SVC

svm_models = []

for i in range(len(bags_X)):
    svm = SVC(kernel='linear', probability=False)
    svm.fit(bags_X[i], bags_y[i])
    svm_models.append(svm)

In [20]:
import random

X_test_combined = []
y_test_combined = []

for i in range(len(X_dm_test)):
    # find all matching GE indices
    matching_indices = [j for j in range(len(X_ge_test)) if y_ge_test[j] == y_dm_test[i]]
    
    # randomly pick one
    j = random.choice(matching_indices)
    
    combined = np.concatenate((X_ge_test[j], X_dm_test[i]))
    
    X_test_combined.append(combined)
    y_test_combined.append(y_dm_test[i])

X_test_combined = np.array(X_test_combined)
y_test_combined = np.array(y_test_combined)

In [21]:
print(X_test_combined.shape)
print(y_test_combined.shape)

(43, 80)
(43,)


In [22]:
pred = model.predict(X_test_combined)

In [23]:
import numpy as np

svm_preds = []

for model in svm_models:
    pred = model.predict(X_test_combined)
    svm_preds.append(pred)

svm_preds = np.array(svm_preds)   # shape: (n_bags, n_samples)

In [24]:
from scipy.stats import mode

final_pred_svm = mode(svm_preds, axis=0).mode.flatten()

In [25]:
from sklearn.metrics import accuracy_score, classification_report

print("=== SVM (Bagging + Ensemble) ===")
print("Accuracy:", accuracy_score(y_test_combined, final_pred_svm))
print(classification_report(y_test_combined, final_pred_svm))

=== SVM (Bagging + Ensemble) ===
Accuracy: 0.7906976744186046
              precision    recall  f1-score   support

           0       0.83      0.71      0.77        21
           1       0.76      0.86      0.81        22

    accuracy                           0.79        43
   macro avg       0.80      0.79      0.79        43
weighted avg       0.80      0.79      0.79        43



In [28]:
with open("../results/bagging_results.txt", "a") as f:
    
    f.write("\n\n=== SVM (Bagging + Ensemble) ===\n")
    f.write("Accuracy: 0.7907\n")
    f.write("""
precision    recall  f1-score   support

0       0.83      0.71      0.77      21
1       0.76      0.86      0.81      22

accuracy                           0.79      43
macro avg       0.80      0.79      0.79      43
weighted avg    0.80      0.79      0.79      43
""")

In [29]:
from sklearn.ensemble import RandomForestClassifier

rf_models = []

for i in range(len(bags_X)):
    rf = RandomForestClassifier(n_estimators=100, random_state=i)
    rf.fit(bags_X[i], bags_y[i])
    rf_models.append(rf)

In [30]:
rf_preds = []

for model in rf_models:
    pred = model.predict(X_test_combined)
    rf_preds.append(pred)

rf_preds = np.array(rf_preds)

In [31]:
from scipy.stats import mode

final_pred_rf = mode(rf_preds, axis=0).mode.flatten()

In [32]:
print("=== Random Forest (Bagging + Ensemble) ===")
print("Accuracy:", accuracy_score(y_test_combined, final_pred_rf))
print(classification_report(y_test_combined, final_pred_rf))

=== Random Forest (Bagging + Ensemble) ===
Accuracy: 0.7906976744186046
              precision    recall  f1-score   support

           0       1.00      0.57      0.73        21
           1       0.71      1.00      0.83        22

    accuracy                           0.79        43
   macro avg       0.85      0.79      0.78        43
weighted avg       0.85      0.79      0.78        43



In [34]:
with open("../results/bagging_results.txt", "a") as f:
    f.write("\n\n=== Random Forest (Bagging + Ensemble) ===\n")
    f.write("Accuracy: YOUR_RF_ACCURACY\n")
    f.write("""=== Random Forest (Bagging + Ensemble) ===
Accuracy: 0.7906976744186046
              precision    recall  f1-score   support

           0       1.00      0.57      0.73        21
           1       0.71      1.00      0.83        22

    accuracy                           0.79        43
   macro avg       0.85      0.79      0.78        43
weighted avg       0.85      0.79      0.78        43""")

In [35]:
from xgboost import XGBClassifier

xgb_models = []

for i in range(len(bags_X)):
    xgb = XGBClassifier(eval_metric='logloss')
    xgb.fit(bags_X[i], bags_y[i])
    xgb_models.append(xgb)

In [36]:
xgb_preds = []

for model in xgb_models:
    pred = model.predict(X_test_combined)
    xgb_preds.append(pred)

xgb_preds = np.array(xgb_preds)

In [37]:
final_pred_xgb = mode(xgb_preds, axis=0).mode.flatten()

In [38]:
print("=== XGBoost (Bagging + Ensemble) ===")
print("Accuracy:", accuracy_score(y_test_combined, final_pred_xgb))
print(classification_report(y_test_combined, final_pred_xgb))

=== XGBoost (Bagging + Ensemble) ===
Accuracy: 0.813953488372093
              precision    recall  f1-score   support

           0       0.93      0.67      0.78        21
           1       0.75      0.95      0.84        22

    accuracy                           0.81        43
   macro avg       0.84      0.81      0.81        43
weighted avg       0.84      0.81      0.81        43



In [40]:
with open("../results/bagging_results.txt", "a") as f:
    f.write("\n\n=== XGBoost (Bagging + Ensemble) ===\n")
    f.write("Accuracy: YOUR_XGB_ACCURACY\n")
    f.write("""=== XGBoost (Bagging + Ensemble) ===
Accuracy: 0.813953488372093
              precision    recall  f1-score   support

           0       0.93      0.67      0.78        21
           1       0.75      0.95      0.84        22

    accuracy                           0.81        43
   macro avg       0.84      0.81      0.81        43
weighted avg       0.84      0.81      0.81        43""")